In [ ]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples = 1000
years_experience = np.random.normal(5, 2, n_samples)
performance_score = np.random.normal(70, 10, n_samples)
gender = np.random.choice(['Male', 'Female'], n_samples)

# Create DataFrame
data = pd.DataFrame({
    'YearsExperience': years_experience,
    'PerformanceScore': performance_score,
    'Gender': gender
})

# Introduce a strong bias: make promotions much more likely for males
promotion_bias = (data['Gender'] == 'Male').astype(int)
promotion = (
    (data['YearsExperience'] * 0.3) +
    (data['PerformanceScore'] * 0.4) +
    (promotion_bias * 3) +  # Adding very strong bias towards males
    np.random.normal(0, 1, n_samples)
)

# Convert promotion scores to binary outcome
data['Promotion'] = (promotion > np.percentile(promotion, 70)).astype(int)  # Promotion for top 30%

# Convert categorical variable to numeric
data['Gender'] = data['Gender'].map({'Male': 1, 'Female': 0})

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Separate features and target
X = data.drop(columns='Promotion')
y = data['Promotion']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
report = classification_report(y_test, y_pred, output_dict=True)

print("Promotion Prediction Model:")
print(f"Accuracy: {report['accuracy']:.2f}")
print(f"Male Promotion Rate: {report['1']['precision']:.2f}")
print(f"Female Promotion Rate: {report['0']['precision']:.2f}")

Promotion Prediction Model:
Accuracy: 0.94
Male Promotion Rate: 0.95
Female Promotion Rate: 0.94


In [ ]:
!pip install aif360

In [ ]:
df_train = X_train.copy()
df_train['Promotion'] = y_train.values
dataset_train = BinaryLabelDataset(df=df_train, label_names=['Promotion'], protected_attribute_names=['Gender'])

In [ ]:
# Train a Logistic Regression model
model = LogisticRegression()
model.fit(dataset_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
report = classification_report(y_test, y_pred, output_dict=True)

print("Promotion Prediction Model:")
print(f"Accuracy: {report['accuracy']:.2f}")
print(f"Male Promotion Rate: {report['1']['precision']:.2f}")
print(f"Female Promotion Rate: {report['0']['precision']:.2f}")

In [ ]:
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric

# Prepare the data for AIF360
df_test = X_test.copy()
df_test_pred = df_test.copy()
df_test['Promotion'] = y_test.values
df_test_pred['Promotion'] = y_pred

# Convert to AIF360 dataset
dataset = BinaryLabelDataset(df=df_test, label_names=['Promotion'], protected_attribute_names=['Gender'])
dataset_pred = BinaryLabelDataset(df=df_test_pred, label_names=['Promotion'], protected_attribute_names=['Gender'])

# Privileged and unprivileged groups
privileged_groups = [{'Gender': 1}]
unprivileged_groups = [{'Gender': 0}]

metric = ClassificationMetric(
    dataset,
    dataset_pred,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print('Disparate Impact:', metric.disparate_impact())
print('Statistical Parity Difference:', metric.statistical_parity_difference())
print('Equal Opportunity Difference:', metric.equal_opportunity_difference())
print('Average Odds Difference:', metric.average_odds_difference())

Disparate Impact: 0.37305946978743726
Statistical Parity Difference: -0.2339989302906044
Equal Opportunity Difference: 0.08651026392961869
Average Odds Difference: 0.03810807314127993


In [ ]:
from aif360.algorithms.preprocessing import Reweighing
RW = Reweighing(unprivileged_groups=unprivileged_groups,
                privileged_groups=privileged_groups)
dataset_transf_train = RW.fit_transform(dataset_train)

In [ ]:
df_trans_train = dataset_transf_train.convert_to_dataframe()[0].drop(columns=['Promotion'])
df_trans_train = df_trans_train * dataset_transf_train.convert_to_dataframe()[1]['instance_weights'].reshape(-1, 1)

In [ ]:
# Train a Logistic Regression model
model = LogisticRegression()
model.fit(df_trans_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
report = classification_report(y_test, y_pred, output_dict=True)

print("Promotion Prediction Model:")
print(f"Accuracy: {report['accuracy']:.2f}")
print(f"Male Promotion Rate: {report['1']['precision']:.2f}")
print(f"Female Promotion Rate: {report['0']['precision']:.2f}")

Promotion Prediction Model:
Accuracy: 0.73
Male Promotion Rate: 0.83
Female Promotion Rate: 0.73


In [ ]:
# Prepare the data for AIF360
df_test = X_test.copy()
df_test_pred = df_test.copy()
df_test['Promotion'] = y_test.values
df_test_pred['Promotion'] = y_pred

# Convert to AIF360 dataset
dataset = BinaryLabelDataset(df=df_test, label_names=['Promotion'], protected_attribute_names=['Gender'])
dataset_pred = BinaryLabelDataset(df=df_test_pred, label_names=['Promotion'], protected_attribute_names=['Gender'])

# Privileged and unprivileged groups
privileged_groups = [{'Gender': 1}]
unprivileged_groups = [{'Gender': 0}]

metric = ClassificationMetric(
    dataset,
    dataset_pred,
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

print('Disparate Impact:', metric.disparate_impact())
print('Statistical Parity Difference:', metric.statistical_parity_difference())
print('Equal Opportunity Difference:', metric.equal_opportunity_difference())
print('Average Odds Difference:', metric.average_odds_difference())

Disparate Impact: 0.8987341772151899
Statistical Parity Difference: -0.002139418791228382
Equal Opportunity Difference: 0.042521994134897365
Average Odds Difference: 0.024937467655683977
